# Classifying time series

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tommoral/24-sacl-ai-4-sciences/blob/main/session-3/02-classifying-signals.ipynb)

Authors: Thomas Moreau

<div align='center'>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Cloaked_in_red.jpg/500px-Cloaked_in_red.jpg" width="400" style="display:inline-block;"/> <span minimum-width=150 style="display:inline-block;"></span><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/2f/Pluto-Charon_system-new.gif/330px-Pluto-Charon_system-new.gif" width="350" style="display:inline-block;"/>
</div>

## Variable stars

Most stars emit light steadily in time, but a small fraction of them has a variable light curve: light emission versus time. We call them variable stars. The light curves are usually periodic and highly regular. There are essentially two reasons why light emission can vary. First, the star itself can be oscillating, so its light emission varies in time. Second, the star that seems a single point at Earth (because of our large distance) is actually a binary system: two stars that orbit around their common center of gravity. When the orbital plane is parallel to our line of view, the stars eclipse each other periodically, creating a light curve with a charateristic signature. Identifying, classifying, and analyzing variable stars are hugely important for calibrating distances, and making these analyses automatic will be crucial in the upcoming sky survey projects such as LSST.

In this notebook, we consider an extract of 2912 stars from the [EROS-1 catalog](http://eros.in2p3.fr/) with 4 types of variable light curves:
- [Eclipsing binary](http://en.wikipedia.org/wiki/Binary_star#Eclipsing_binaries),
- [Cepheid](http://en.wikipedia.org/wiki/Cepheid_variable),
- [RR-Lyrae](http://en.wikipedia.org/wiki/RR_Lyrae_variable),
- [Mira](http://en.wikipedia.org/wiki/Mira_variable).

Our goal is to learn a function that assign an observed variable star to one of these 4 classes. The observations are composed of the light curves in two frequency bands (red and blue), as well as static variables. 

Let us for load the data:

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = "data/variable_stars.parquet"
URL_REPO = "https://github.com/bcg-advance-ml-training/materials/raw/main/02-time-series/"

if Path(DATA_FILE).exists():
    X_df = pd.read_parquet(DATA_FILE)
else:
    X_df = pd.read_parquet(f"{URL_REPO}{DATA_FILE}")

y = X_df['type']
X_df = X_df.drop(columns='type')


In [ ]:
X_df.head()

Let's take a quick look at one light curve per class to build intuition before modelling.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, cls in zip(axes, y.unique()):
    idx = y[y == cls].index[0]
    star = X_df.loc[idx]
    # Fold on the known period to show one representative cycle
    phase = (star['time_points_b'] % star['period']) / star['period']
    ax.scatter(phase, star['light_points_b'], s=5, alpha=0.6)
    ax.invert_yaxis()  # astronomical convention: lower magnitude = brighter
    ax.set_title(cls, fontsize=12)
    ax.set_xlabel("Phase")
axes[0].set_ylabel("Magnitude (B band)")
plt.suptitle("One example per class — phase-folded light curve", y=1.02)
plt.tight_layout()
plt.show()

There are a few informative static variables in this dataframe:

<ul>
    <li> <code><b><span style="color:black">period</span></b></code>: the estimated period of the light curve.
    <li> <code><b>magnitude_b, magnitude_r</b></code>: The average apparent luminosity of the star (in two frequency bands). <a href="http://www.astro-tom.com/technical_data/magnitude_scale.htm">Magnitude</a> is a logarithmic measure, and the higher it is, the lower the apparent luminosity is.
    <li> <code><b><span style="color:black">log_p_not_variable</span></b></code>: Logarithm of the estimated probability that the star is stable.
    <li> <code><b><span style="color:black">sigma_flux_b, sigma_flux_r</span></b></code>: The square root of the total variance of the light measurements (indicating the amplitude of the variability).
    <li> <code><b><span style="color:red">type</span></b></code>: The label to predict.
</ul> 

Let us first start with a simple model based on these static features.
We start by splitting the data between train and test:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_df, y, random_state=20240707)

Now we will start with a dummy method, to see how hard this problem is:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score

clf = DummyClassifier(
    strategy="most_frequent"
).fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("Balanced accuracy:", balanced_accuracy_score(y_test, clf.predict(X_test)))

<div class="alert alert-success">
    <b>EXERCISE</b>:
    Using what you learn yesterday, can you create a pipeline with the following steps
     <ul>
         <li>A column transformer that selects columns <code>'magnitude_b', 'magnitude_r', 'sigma_flux_b', 'sigma_flux_r', 'log_p_not_variable'</code>,</li>
         <li>A <code>RandomForestClassifier</code></li>
    </ul>
    
 <i>Hint:</i> To select the columns without modifying them, you can use <code>'passthrough'</code> as a <code>Transformer</code> in the <code>ColumnTransformer</code>.
</div>

Solution: `solutions/classif-1_static-pipeline.py`


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier

clf = make_pipeline(
    ...
).fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("Balanced accuracy:", balanced_accuracy_score(y_test, clf.predict(X_test)))


### Constructing a classifier based on the light curves

We will now turn toward building a classifier based on the raw light curves, that are stored in `time_points_r/b` and `light_points_r/b` columns of our dataset.

A first step when working with signals is usually to display them and take a look at their properties.

In [ ]:
import matplotlib.pyplot as plt

for idx in [10, 11, 100]:
    star = X_df.iloc[idx]
    print(f"Number of time points for star {idx}: {len(star['time_points_b'])}")

    plt.scatter(star['time_points_b'], star['light_points_b'])

<div class="alert alert-info">
    <b>QUESTION</b>:
     <ul>
         <li>What can you say about these time-series?
             <i>Are they regular? Comparable? Noisy?</i></li>
         <li>How would you compare them?</li>
    </ul>
</div>


### Aligning and resampling the signals

The benefit of working with global features is that they can often be computed even with unaligned time-series.
However, these features can fail to catpure some interesting notion of similarity between the samples.

We will now explore how to align and resample the data.

Here, we have data that is periodic, and we are informed about the period by the column `period`.

In [ ]:
X_df['period']

We can thus look at the data on one period:

In [ ]:
import matplotlib.pyplot as plt

for idx in [10, 11, 100]:
    star = X_df.iloc[idx]
    print(f"Number of time points for star {idx}: {len(star['time_points_b'])}")

    plt.scatter((star['time_points_b'] % star['period']) / star['period'], star['light_points_b'])

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
         <li>How would you modify this code to "align" the time series?</li>
         <li>How would you modify this code to cope with the fact that the sample are not aligned?</li>
    </ul>
    
   *Hint:* resampling non-uniform points can be performed with the [`scipy.interpolate.interp1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html) function.
</div>

Solution: `solutions/classif-2_align_periodic_signals.py`

We can now do a `scikit-learn` pipeline directly on the resampled light curves using the `FunctionTransformer`.
`FunctionTransformer` allow you to use any kind of function transforming your data in a `scikit-learn` pipeline.
The caveats of this tranformer is that it cannot do anything during the `fit` part of the pipeline.
To create a new `Transformer` that resample the data, you can use:

In [ ]:
import numpy as np
from scipy import interpolate
from sklearn.preprocessing import FunctionTransformer

t = np.linspace(0, 1, 100)
def resample(star):
    t_i = (star['time_points_b'] % star['period']) / star['period']
    interp = interpolate.interp1d(
        t_i, star['light_points_b'],
        kind='linear', bounds_error=False, fill_value="extrapolate"
    )
    return pd.Series({k: v for k, v in zip(t, interp(t))})

X_train_t = FunctionTransformer(lambda X: X.apply(resample, axis=1)).transform(X_train)
X_train_t

<div class="alert alert-success">
    <b>EXERCISE</b>:
    Extend your static pipeline with the resampled time-series.
</div>

Solution: `solutions/classif-3_full-pipeline.py`

### Using time-series-native classifiers with `tslearn` and `aeon`

As covered in the slides, `tslearn` focuses on **DTW-based** methods (k-NN, SVM kernels) — good intuition builders but slow at scale. `aeon` covers more ground and includes state-of-the-art **convolution-based** classifiers (ROCKET, MiniRocket) that train on the full dataset in seconds. See the full list of classifiers in AEON [here](https://www.aeon-toolkit.org/en/stable/api_reference/classification.html).

In [ ]:
# !pip install -U tslearn aeon

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from tslearn.neighbors import KNeighborsTimeSeriesClassifier

# DTW computation is O(n * T^2) where T is the series length.
# We train on a subset of 500 samples to keep this cell fast.
clf_knn_dtw = make_pipeline(
    FunctionTransformer(lambda X: X.apply(resample, axis=1)),
    KNeighborsTimeSeriesClassifier(n_neighbors=5, metric="dtw", n_jobs=-1)
).fit(X_train[:500], y_train[:500])

y_pred_knn = clf_knn_dtw.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_knn))
print("(trained on 500 samples only — DTW is slow at scale)")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from tslearn.svm import TimeSeriesSVC

clf_svc_dtw = make_pipeline(
    FunctionTransformer(lambda X: X.apply(resample, axis=1)),
    TimeSeriesSVC()
).fit(X_train[:100], y_train[:100])

y_pred_svc = clf_svc_dtw.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_svc))
print("(trained on 100 samples only — SVC is slow at scale)")

#### ROCKET and MiniRocket (aeon)

**ROCKET** (*RandOm Convolutional KErnel Transform*) applies thousands of random convolutional kernels to each series and extracts two features per kernel (max activation and proportion of positive values). A simple linear classifier (RidgeCV) is then trained on these features.

- Trains on the **full dataset** in seconds — no subset needed
- State-of-the-art accuracy on most TS classification benchmarks
- **MiniRocket** is a refined variant: faster, near-deterministic, and slightly more accurate

In [ ]:
import numpy as np
from aeon.classification.convolution_based import RocketClassifier

# Resample to a uniform grid — shape (n_samples, 100)
X_train_rs = X_train.apply(resample, axis=1).to_numpy()
X_test_rs = X_test.apply(resample, axis=1).to_numpy()

clf_rocket = RocketClassifier(n_kernels=500, n_jobs=-1)
clf_rocket.fit(X_train_rs, y_train)  # full training set

y_pred = clf_rocket.predict(X_test_rs)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

In [ ]:
from aeon.classification.convolution_based import MiniRocketClassifier

clf_minirocket = MiniRocketClassifier(n_jobs=-1)
clf_minirocket.fit(X_train_rs, y_train)  # reuses the resampled arrays from above

y_pred = clf_minirocket.predict(X_test_rs)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

#### Combining MiniRocket features with static features

The `RocketClassifier` / `MiniRocketClassifier` wrappers only see the time series. But the static features (period, magnitude, sigma_flux…) are highly informative. We can combine both inside a sklearn `Pipeline` using a `ColumnTransformer` with two branches — the `ColumnTransformer` handles the concatenation automatically.

```
                                      ┌─ StandardScaler ──────────────────────────────────┐
X_train (DataFrame)  →  ColumnTransformer                                                  ├─ hstack  →  RidgeCV
                                      └─ resample_and_reshape  →  MiniRocket (transformer)┘
```

The key difference from the `MiniRocketClassifier` wrapper: we use `MiniRocket` (the **transformer**), which requires 3-D input `(n_samples, n_channels, n_timepoints)`, and pair it with `RidgeClassifierCV` explicitly.

In [ ]:
import numpy as np
from aeon.transformations.collection.convolution_based import MiniRocket
from sklearn.compose import make_column_transformer
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

STATIC_COLS = ['magnitude_b', 'magnitude_r', 'sigma_flux_b', 'sigma_flux_r', 'log_p_not_variable']
TS_COLS = ['time_points_b', 'light_points_b', 'period']

def resample_and_reshape(X):
    """Resample each row to a uniform grid and reshape to 3D (n_samples, n_channels, n_timepoints)."""
    return X.apply(resample, axis=1).to_numpy()[:, np.newaxis, :]

clf_combined = make_pipeline(
    make_column_transformer(
        (StandardScaler(), STATIC_COLS),
        (make_pipeline(FunctionTransformer(resample_and_reshape), MiniRocket()), TS_COLS),
    ),
    RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)),
).fit(X_train, y_train)

y_pred_combined = clf_combined.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_combined))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_combined))

Both `tslearn` and `aeon` implement many more algorithms (clustering, regression, anomaly detection). All classifiers above require signals of the **same length**, which is why the resampling step was necessary.

### Extracting global features with `tsfresh`

As the slides noted, `tsfresh` has a **complex API** — but `aeon` wraps it in a sklearn-compatible transformer. We will use the `aeon` wrapper directly in a pipeline rather than the raw `tsfresh` API.

For reference, the raw `tsfresh` API expects a long-form `pd.DataFrame` with `id` (which signal) and `time` (when) columns — convenient for irregular or variable-length series.

In [ ]:
# !pip install -U tsfresh aeon

With this format, it is convenient to represent non-uniform time-series, with variable lenghts.
Let us create these `DataFrame` for the train and test:

In [ ]:
df_ts_train_b = pd.DataFrame([
    {'id': idx, 'time': t, 'light_b': v}
    for idx, row in X_train.iterrows()
    for t, v in zip(row['time_points_b'], row['light_points_b'])
])
df_ts_train_b

`Tsfresh` provides a convenient helper to extract many possible features at once.

In [ ]:
from tsfresh import extract_relevant_features
extracted_features = extract_relevant_features(df_ts_train_b, y_train, column_id="id", column_sort="time")
extracted_features

In [ ]:
extracted_features.shape

The `scikit-learn` like API provided by tsfresh is complicated to use.
Conveniently, `aeon` exposes an easy to use interface for TSFresh that allow working with pipelines:

In [ ]:
from sklearn import config_context
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier

from aeon.transformations.collection.feature_based import TSFresh


# this cell takes a few minutes to run
with config_context(transform_output='pandas'):

    clf_tsfresh = make_pipeline(
        make_column_transformer(
            ('passthrough', ['magnitude_b', 'magnitude_r', 'sigma_flux_b', 'sigma_flux_r', 'log_p_not_variable']),
            (make_pipeline(
                FunctionTransformer(lambda X: X.apply(resample, axis=1)),
                TSFresh()
            ), ['time_points_b', 'light_points_b', 'period'])
        ),
        RandomForestClassifier()
    ).fit(X_train, y_train)

    y_pred_tsfresh = clf_tsfresh.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_tsfresh))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_tsfresh))

### Mantis — a Time-Series Classification Foundation Model

[**Mantis**](https://github.com/vfeofanov/mantis) (Huawei Noah's Ark Lab, 2025) is a pre-trained 8 M-parameter transformer for TS classification. It was pre-trained on a large corpus of UCR/UEA datasets and can be used **zero-shot** or fine-tuned on a new dataset.

Key API: `MantisTrainer` is sklearn-compatible (`fit` / `predict` / `predict_proba`).

- Input shape: `(n_samples, n_channels, n_timepoints)` — timepoints must be a multiple of 32; the authors recommend 512.
- Works on CPU (slow) or GPU.

In [ ]:
# !pip install mantis-tsfm

In [ ]:
import torch
import torch.nn.functional as F
from mantis.architecture import Mantis8M
from mantis.trainer import MantisTrainer

# Load pre-trained weights from HuggingFace (cached after first download)
network = Mantis8M(device='cpu').from_pretrained("paris-noah/Mantis-8M")
mantis = MantisTrainer(device='cpu', network=network)

def mantis_transform(X):
    """Resize (n_samples, n_timepoints) → (n_samples, 1, 512), then apply Mantis's transform to extract features."""
    X = X.apply(resample, axis=1).to_numpy()[:, np.newaxis, :]
    t = torch.tensor(X, dtype=torch.float)  # (n, 1, T)
    # Mantis was pre-trained on series of length 512 (must be a multiple of 32)
    x = F.interpolate(t, size=512, mode='linear', align_corners=False).numpy()
    return mantis.transform(x)

clf_mantis = make_pipeline(
    make_column_transformer(
        (StandardScaler(), STATIC_COLS),
        (make_pipeline(FunctionTransformer(mantis_transform)), TS_COLS),
    ),
    RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)),
).fit(X_train, y_train)

y_pred_mantis = clf_mantis.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_mantis))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_mantis))

### Summary: comparing approaches

Let us collect balanced accuracy scores across all approaches on the same test set.

> **Note:** KNN-DTW was trained on 500 samples and SVC on 100 — DTW/kernel computation is O(n·T²), so their scores are not directly comparable to the full-dataset methods.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier

results = {}

# Baseline
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
results["Dummy (most frequent)"] = balanced_accuracy_score(y_test, dummy.predict(X_test))

# Static features only
results["Static features + RandomForest"] = balanced_accuracy_score(y_test, y_pred_static)

# Distance-based — 100 samples only, not directly comparable to others
results["KNN-DTW  ⚠️ 500 samples"] = balanced_accuracy_score(y_test, y_pred_knn)
results["SVC-DTW  ⚠️ 100 samples"] = balanced_accuracy_score(y_test, y_pred_svc)

# Convolution-based (full training set)
results["ROCKET (aeon)"] = balanced_accuracy_score(y_test, clf_rocket.predict(X_test_rs))
results["MiniRocket (aeon)"] = balanced_accuracy_score(y_test, clf_minirocket.predict(X_test_rs))
results["MiniRocket + static features"] = balanced_accuracy_score(y_test, y_pred_combined)

# Feature extraction
results["tsfresh + RandomForest"] = balanced_accuracy_score(y_test, y_pred_tsfresh)

# Foundation model
results["Mantis (linear-probe)"] = balanced_accuracy_score(y_test, y_pred_mantis)

summary = pd.DataFrame.from_dict(results, orient="index", columns=["Balanced accuracy"])
summary = summary.sort_values("Balanced accuracy", ascending=False)
summary.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.25, vmax=1.0)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion matrix for the best convolution-based model
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_mantis,
    ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion matrix — Mantis")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_combined,
    ax=axes[1], colorbar=False
)
axes[1].set_title("Confusion matrix — MiniRocket + static")
plt.tight_layout()
plt.show()